In [0]:
from pyspark.sql.functions import col, count, round as spark_round, min as spark_min, max as spark_max, from_unixtime
from pyspark.sql.types import DoubleType, LongType, IntegerType

# --- Read the OpenCellID CSV (no header; 14 columns) ---
raw = (
    spark.read.format("csv")
    .option("header", "false")
    .load("/Volumes/cmegdemos_catalog/network_analytics_enablement/raw_data/310.csv.gz")
)

# OpenCellID schema: 14 columns
column_names = ["radio", "mcc", "net", "area", "cell", "unit",
                "lon", "lat", "range_m", "samples", "changeable",
                "created", "updated", "avg_signal"]
df = raw.toDF(*column_names).select(
    col("radio"),
    col("mcc").cast(IntegerType()),
    col("net").cast(IntegerType()),
    col("area").cast(IntegerType()),
    col("cell").cast(LongType()),
    col("unit").cast(IntegerType()),
    col("lon").cast(DoubleType()),
    col("lat").cast(DoubleType()),
    col("range_m").cast(IntegerType()),
    col("samples").cast(IntegerType()),
    col("changeable").cast(IntegerType()),
    from_unixtime(col("created").cast(LongType())).alias("created_ts"),
    from_unixtime(col("updated").cast(LongType())).alias("updated_ts"),
    col("avg_signal").cast(IntegerType()),
)

total = df.count()
print(f"Total records in file (MCC 310 = USA): {total:,}")
print(f"\n--- Filtering to Seattle metro area (lat 47.4-47.8, lon -122.5 to -122.1) ---")

# Filter to Seattle bounding box
seattle = df.filter(
    (col("lat").between(47.4, 47.8)) & (col("lon").between(-122.5, -122.1))
)
seattle.createOrReplaceTempView("seattle_towers")

seattle_count = seattle.count()
print(f"Seattle-area cell tower records: {seattle_count:,}")
print()
display(seattle.limit(15))

In [0]:
# MNC reference for MCC 310 (USA):
# 260 = T-Mobile, 410 = AT&T, 120 = Sprint, 480 = Verizon (iWireless)

mnc_labels = {260: "T-Mobile", 410: "AT&T", 120: "Sprint", 480: "Verizon (iWireless)", 150: "AT&T"}

print("=== Seattle Cell Tower Breakdown ===")
df_breakdown = spark.sql("""
    SELECT 
        radio,
        net AS mnc,
        COUNT(*) AS record_count,
        COUNT(DISTINCT cell) AS unique_cell_ids,
        ROUND(AVG(range_m), 0) AS avg_range_m,
        ROUND(AVG(samples), 1) AS avg_samples
    FROM seattle_towers
    GROUP BY radio, net
    ORDER BY record_count DESC
""")
display(df_breakdown)

In [0]:
# Show sample records with carrier labels
print("=== Sample Seattle Tower Records ===")
display(spark.sql("""
    SELECT 
        radio,
        net AS mnc,
        CASE net 
            WHEN 260 THEN 'T-Mobile' 
            WHEN 410 THEN 'AT&T' 
            ELSE CONCAT('MNC-', net) 
        END AS carrier,
        area AS lac_tac,
        cell AS cell_id,
        lon, lat, range_m, samples,
        created_ts, updated_ts
    FROM seattle_towers
    ORDER BY samples DESC
    LIMIT 20
"""))